In [1]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# 1. PROJECT CONFIGURATION
PROJECT_FOLDER = Path(r"D:\py_de_ai_project")
LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"
PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"

# Create folders if they don't exist
for folder in [LANDING, PROCESSING, ARCHIVE, ERROR, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)
# Load API key and initialize AI client
load_dotenv(PROJECT_FOLDER / ".env")
GEMINI_API_KEY="AQ.Ab8RN6Ioi1aTRxsJhYVSGfT-Q70JGv-2KS8ULBwFJy4l_N-zHQ"
client = genai.Client(api_key=os.getenv(GEMINI_API_KEY))
MODEL = "gemini-3.5-flash-lite"

print("Folders created and AI client initialized!")

Folders created and AI client initialized!


In [24]:
from google import genai

# Replace with your actual Gemini API key
client = genai.Client(api_key="AQ.Ab8RN6Ioi1aTRxsJhYVSGfT-Q70JGv-2KS8ULBwFJy4l_N-zHQ")

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain Python functions in simple words."
)

print(response.text)

Think of a Python function like a **blender**. 

1. You put something inside it (**input**).
2. It does a specific job (**the code inside**).
3. It pours out the final result (**output**).

Once you have a blender, you don’t have to build a new one every time you want juice—you just push the button. Functions work the same way: **you write the code once, and you can use it as many times as you want.**

---

### A Simple Example

Here is a function that says hello to someone:

```python
def say_hello(name):
    return "Hello, " + name + "!"
```

Here is what those words mean:
* **`def`**: Short for "define." It just tells Python, *"Hey, I am building a new machine."*
* **`say_hello`**: The name of the machine. You can name it whatever you want.
* **`(name)`**: The input (like the fruit you put in the blender). 
* **`return`**: The output. It hands the final result back to you.

---

### How to use it (Pressing the button)

Just writing the function doesn't make it do anything. You have 

In [32]:
# 2. DEFINE FUNCTION
def load_dataframe(file_path):
    extension = file_path.suffix.lower()

    if extension == ".csv":
        return pd.read_csv(file_path)
    if extension == ".json":
        return pd.read_json(file_path)
    if extension == ".xlsx":
        return pd.read_excel(file_path)
    if extension == ".txt":
        return pd.read_csv(file_path, sep="|")

    print(f"Unsupported file type: {extension}")
    return None

# TEST IT
test_file = LANDING / "employee.csv" # Change to your actual file name
df = load_dataframe(test_file)

print("\n--- Testing load_dataframe ---")
print(df.head())


--- Testing load_dataframe ---
   BranchID  EmployeeID FirstName    Hiredate  LastName  ManagerID  \
0       130      1876.0    Pamela  05-03-2020    Thomas        NaN   
1       167      1877.0   Michael  30-12-2020      King        NaN   
2       114      1878.0     Linda  18-12-2022     Adams        NaN   
3       253         NaN    Laurie  03-09-2017  Mcintyre        NaN   
4       108         NaN  Danielle  13-04-2016    Holmes        NaN   

            Position  
0  Assistant Manager  
1            Manager  
2             Manger  
3            Manager  
4            Manager  


In [16]:
# 4. DEFINE FUNCTION
import pandas as pd
import json
import os

# Step 1: Load your input file
file_path = r"D:\py_de_ai_project\landing\employee.csv"

print("File exists:", os.path.exists(file_path))

df = pd.read_csv(file_path)
print(df.head())


# Step 2: Get column data types
data_types = {
    column: str(dtype)
    for column, dtype in df.dtypes.items()
}

# Step 3: Create a simple column profile
column_profile = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

def build_prompts(file_name, df, data_types, column_profile):
    sample_json = df.head(10).astype(str).to_json(orient="records", indent=2, force_ascii=False)
    columns = df.columns.tolist()
    types_json = json.dumps(data_types, indent=2, ensure_ascii=False)
    profile_text = column_profile.to_string(index=False)

    classification_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Data governance domain, PII/confidential tags, automated data quality rules & checks. Return valid JSON."""

    schema_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Target analytical DDL schema, primary/foreign key evidence & data quality constraints. Return valid JSON."""
    return classification_prompt, schema_prompt

# TEST IT
print("\n--- Testing build_prompts ---")
c_prompt, s_prompt = build_prompts(test_file.name, df, data_types, column_profile)

print("Classification Prompt preview:")
print(c_prompt[:500] + "...\n[TRUNCATED FOR DISPLAY]")

File exists: True
   BranchID  EmployeeID FirstName    Hiredate  LastName  ManagerID  \
0       130      1876.0    Pamela  05-03-2020    Thomas        NaN   
1       167      1877.0   Michael  30-12-2020      King        NaN   
2       114      1878.0     Linda  18-12-2022     Adams        NaN   
3       253         NaN    Laurie  03-09-2017  Mcintyre        NaN   
4       108         NaN  Danielle  13-04-2016    Holmes        NaN   

            Position  
0  Assistant Manager  
1            Manager  
2             Manger  
3            Manager  
4            Manager  

--- Testing build_prompts ---
Classification Prompt preview:
File: landingemployees.csv
Columns: ["BranchID", "EmployeeID", "FirstName", "Hiredate", "LastName", "ManagerID", "Position"]
Types:
{
  "BranchID": "int64",
  "EmployeeID": "float64",
  "FirstName": "str",
  "Hiredate": "str",
  "LastName": "str",
  "ManagerID": "float64",
  "Position": "str"
}
Profile:
    Column  Missing Values  Unique Values
  BranchID    

In [11]:
import pandas as pd
import json
import os

# Step 1: Load your input file
file_path = r"D:\py_de_ai_project\landing\employee.csv"

print("File exists:", os.path.exists(file_path))

df = pd.read_csv(file_path)

File exists: True


In [48]:
# 5. DEFINE FUNCTION
def ask_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

# TEST IT
print("\n--- Testing ask_gemini ---")
print("Sending prompt to AI... please wait.")
classification_result = ask_gemini("how to create a API")
print(classification_result)


--- Testing ask_gemini ---
Sending prompt to AI... please wait.
Creating an API (Application Programming Interface)—specifically a **REST API**, which is the most common type—involves a few key steps: planning, choosing your tools, writing the code, and testing it.

Here is a beginner-friendly, step-by-step guide to building your first API using **JavaScript (Node.js & Express)**.

---

### Step 1: Plan Your API
Before writing code, decide what your API will do. Let's create a simple API to manage a list of **books**.

You need to define your **Endpoints** (URLs) and **HTTP Methods**:
*   `GET /books` — Retrieve all books
*   `POST /books` — Add a new book
*   `DELETE /books/:id` — Delete a book by its ID

---

### Step 2: Set Up Your Environment
Make sure you have [Node.js](https://nodejs.org/) installed on your computer.

1. Open your terminal or command prompt.
2. Create a new folder and move into it:
   ```bash
   mkdir my-first-api
   cd my-first-api
   ```
3. Initialize a Node.j

In [17]:
# =====================================================================
# 6. ASSEMBLE PIPELINE FUNCTIONS & TESTING EXECUTION
# =====================================================================
import pandas as pd
import json
import os
def process_file(source_file):
    """
    Executes the end-to-end pipeline lifecycle for an incoming dataset:
    1. Locks file: moves from landing/ -> data/processing/
    2. Loads into Pandas dataframe dynamically (.csv, .json, .xlsx, .txt)
    3. Profiles dataset metrics & saves _profile.csv
    4. Builds governance and target DDL schema prompts
    5. Queries Gemini AI for dual passes (Governance + DDL Modeling)
    6. Persists _classification.json, _schema.json, and _processed.csv
    7. Safely archives source file to data/archive/
    """
    file_name = source_file.name
    processing_file = PROCESSING / file_name

    print(f"\n{'='*55}")
    print(f"PROCESSING: {file_name} ({source_file.stat().st_size} bytes)")
    print(f"{'='*55}")

    # 1. Atomic lock: move file from landing/ to processing/
    shutil.move(str(source_file), str(processing_file))

    # 2. Ingest via dynamic multi-format reader
    df = load_dataframe(processing_file)
    if df is None:
        print(f"FAILED: Could not parse dataframe for {file_name}")
        shutil.move(str(processing_file), str(ERROR / file_name))
        return

    if len(df.columns) == 0:
        print(f"WARNING: Dataset has 0 columns: {file_name}")
        shutil.move(str(processing_file), str(ERROR / file_name))
        return

    # 3. Compute statistical and structural profile
    data_types, column_profile, duplicates = profile_dataframe(df, file_name)

    # 4. Construct AI prompts
    c_prompt, s_prompt = build_prompts(file_name, df, data_types, column_profile)

    # 5. Dual AI Inferences (Governance + DDL Modeling)
    print(f"Requesting AI Data Governance & Classification ({MODEL})...")
    classification_json = ask_gemini(c_prompt)

    print(f"Requesting Target Schema & Key Recommendations ({MODEL})...")
    schema_json = ask_gemini(s_prompt)

    # 6. Persist generated artifacts
    classification_file = OUTPUT / f"{Path(file_name).stem}_classification.json"
    schema_file = OUTPUT / f"{Path(file_name).stem}_schema.json"
    processed_file = OUTPUT / f"{Path(file_name).stem}_processed.csv"

    classification_file.write_text(classification_json, encoding="utf-8")
    schema_file.write_text(schema_json, encoding="utf-8")
    df.to_csv(processed_file, index=False)

    # 7. Move source file to archive
    shutil.move(str(processing_file), str(ARCHIVE / file_name))

    print(f"\nSUCCESS: {file_name}")
    print(f"  Profile:        {OUTPUT / f'{Path(file_name).stem}_profile.csv'}")
    print(f"  Classification: {classification_file}")
    print(f"  Schema:         {schema_file}")
    print(f"  Processed CSV:  {processed_file}")
    print(f"  Archived to:    {ARCHIVE / file_name}")


def main():
    """
    Main batch scanner:
    - Scans landing/ for .csv, .json, .xlsx, .txt
    - Quarantines unsupported formats into error/
    - Passes valid files sequentially to process_file()
    """
    supported_extensions = [".csv", ".json", ".xlsx", ".txt"]
    files = []

    for f in LANDING.iterdir():
        if f.is_file():
            if f.suffix.lower() in supported_extensions:
                files.append(f)
            else:
                print(f"Unsupported file quarantined: {f.name}")
                shutil.move(str(f), str(ERROR / f.name))

    files.sort()
    print(f"\n--- Batch Ingestion Scan ---")
    print(f"Found {len(files)} supported file(s) in {LANDING}:")
    for f in files:
        print(f"  -> {f.name}")

    if not files:
        print("No files available in landing/ to process.")
        return

    for source_file in files:
        process_file(source_file)

    print(f"\nAll files processed successfully! Check {OUTPUT} and {ARCHIVE}.")


print("TESTING: Orchestrator & Batch Execution")

# Option A: Run a single sample file if available in landing/
sample_test = LANDING / "employee.csv"
if sample_test.exists():
    print(f"Testing single file execution: {sample_test.name}")
    process_file(sample_test)
else:
    # Option B: Run full batch scan across landing/ folder
    print("Testing full batch execution across landing/...")
    main()

TESTING: Orchestrator & Batch Execution
Testing single file execution: employee.csv

PROCESSING: employee.csv (166533 bytes)


NameError: name 'load_dataframe' is not defined